In [1]:
import os
import re
import time
from pathlib import Path
from collections import OrderedDict
import xml.etree.ElementTree as ET

# ── Local Configuration for Aristotle's Poetics ────────────────
OUT_DIR = Path("/Users/gcrane/Downloads/gemsite/poetics_site")
OUT_DIR.mkdir(parents=True, exist_ok=True)

NS = {'tei': 'http://www.tei-c.org/ns/1.0'}

TEXT_REGISTRY = {
    "grc_kassel": {
        "path": "/Users/gcrane/github/canonical-greekLit/data/tlg0086/tlg034/tlg0086.tlg034.perseus-grc2.xml",
        "label": "Greek (Kassel, 1965)",
        "class": "greek-text"
    },
    "grc_digi": {
        "path": "/Users/gcrane/github/canonical-greekLit/data/tlg0086/tlg034/tlg0086.tlg034.digicorpus-grc2.xml",
        "label": "Greek (Digital Corpus Variant)",
        "class": "greek-text"
    },
    "eng_fyfe": {
        "path": "/Users/gcrane/github/canonical-greekLit/data/tlg0086/tlg034/tlg0086.tlg034.perseus-eng2.xml",
        "label": "English (W.H. Fyfe, 1927)",
        "class": "english-text"
    },
    "eng_butcher": {
        "path": "/Users/gcrane/github/Poetics2.0/grc/tlg0086.tlg034.butcher1911-eng2.xml",
        "label": "English (S.H. Butcher, 1911)",
        "class": "english-text"
    },
    "eng_bywater": {
        "path": "/Users/gcrane/github/Poetics2.0/grc/tlg0086.tlg034.bywater1909-eng1.xml",
        "label": "English (Ingram Bywater, 1909)",
        "class": "english-text"
    }
}

# ── Streamlined Localized Pop-up Dropdown CSS Framework ───────
PERSEUS_CSS = r"""
* { margin: 0; padding: 0; box-sizing: border-box; }
html, body {
    height: 100%;
    width: 100%;
    overflow: hidden; 
    font-family: "Palatino Linotype", "Book Antiqua", Palatino, Georgia, serif;
    font-size: 13px;
    color: #000;
    background: #fff;
}
a { color: #336699; text-decoration: none; }
a:hover { text-decoration: underline; }

#app-view-root { 
    display: flex;
    flex-direction: column;
    height: 100vh;
    width: 100vw;
    overflow: hidden;
}

/* Header Container Element Bounds */
#header-container { 
    flex-shrink: 0;
    background: #fff; 
    border-bottom: 1px solid #ccc; 
    z-index: 10;
}
#perseus-banner { background: #660000; color: #fff; padding: 6px 15px; display: flex; justify-content: space-between; align-items: center; }
#perseus-banner h1 a { color: #fff; font-size: 16px; font-weight: bold; }
#perseus-banner .doc-title { font-size: 11px; color: #ffccaa; font-weight: bold; }

#nav-bar { background: #ddddcc; border-bottom: 1px solid #999988; padding: 4px 15px; font-size: 11px; display: flex; align-items: center; }
#nav-bar label { background: #660000; color: #fff; padding: 2px 8px; border-radius: 3px; cursor: pointer; font-weight: bold; margin-right: 15px; user-select: none; }

#browse-bar { background: #eeeeee; border-bottom: 1px solid #cccccc; padding: 5px 15px; font-size: 11px; }
.browse-row { margin-bottom: 2px; display: flex; align-items: center; }
.browse-row label { font-weight: bold; width: 110px; color: #555; flex-shrink: 0; }
.browse-items { display: flex; flex-wrap: wrap; gap: 4px; }
.browse-items a { padding: 1px 5px; background: #e0e0d0; color: #336699; border: 1px solid #bbbb99; font-weight: bold; }
.browse-items a.current { background: #660000; color: #fff; border-color: #330000; }

#outer-wrapper { 
    display: flex; 
    width: 100%;
    flex: 1; 
    min-height: 0; 
}

/* Collapsible Left TOC Sidebar */
#sidebar-toc { 
    width: 220px; background: #f5f5ee; border-right: 1px solid #ccccbb; padding: 12px; font-size: 11px; overflow-y: auto; flex-shrink: 0;
    transition: width 0.15s ease, padding 0.15s ease;
}
#sidebar-toc h3 { font-size: 12px; color: #660000; margin-bottom: 6px; border-bottom: 1px solid #ccccbb; padding-bottom: 2px; }
#sidebar-toc ul { list-style: none; }
#sidebar-toc li { margin: 5px 0; }
#sidebar-toc a.current { color: #660000; font-weight: bold; }
#sidebar-toc .toc-sections { display: flex; flex-wrap: wrap; gap: 3px; padding-left: 5px; margin-top: 4px; }
#sidebar-toc .toc-sections a { background: #e0e0d4; padding: 1px 4px; font-size: 10px; border-radius: 2px; color: #333; font-weight: bold; }
#sidebar-toc .toc-sections a:hover { background: #660000; color: #fff; text-decoration: none; }

#toc-toggle { display: none; }
#toc-toggle:checked ~ #app-view-root #outer-wrapper #sidebar-toc { width: 0px; padding: 0px; border-right: none; overflow: hidden; }
#toc-toggle:checked ~ #app-view-root #header-container #nav-bar .hide-lbl { display: none; }
#toc-toggle:not(:checked) ~ #app-view-root #header-container #nav-bar .show-lbl { display: none; }

/* Triple Reading Columns Parallel Container Layout */
#main-container { flex: 1; display: flex; min-width: 0; height: 100%; }
.reading-column { flex: 1; min-width: 0; height: 100%; overflow-y: auto; padding: 15px; border-right: 1px solid #eeeeee; }
.reading-column:last-child { border-right: none; }
.cross-panel { background: #fafafa; }

/* Localized Selection Header Component */
.panel-header { 
    background: #660000; 
    color: #fff; 
    padding: 4px 8px; 
    font-size: 11px; 
    font-weight: bold; 
    margin-bottom: 12px; 
    position: sticky; 
    top: 0; 
    z-index: 5;
    display: flex;
    justify-content: space-between;
    align-items: center;
}

/* Pure CSS Local Popup Select Trigger Elements */
.edition-select-dropdown {
    background: #fff;
    color: #333;
    font-family: inherit;
    font-size: 10px;
    padding: 1px 4px;
    border: 1px solid #ccc;
    border-radius: 2px;
    outline: none;
    cursor: pointer;
}

input[type="radio"] { display: none; }
.version-block { display: none; }

/* ── DOM Flat Sibling Selection Mapping Controls Engine ── */
#f_grc_kassel:checked   ~ #app-view-root #main-container .col1-container .v-grc_kassel,
#f_grc_digi:checked     ~ #app-view-root #main-container .col1-container .v-grc_digi,
#f_eng_fyfe:checked     ~ #app-view-root #main-container .col1-container .v-eng_fyfe,
#f_eng_butcher:checked  ~ #app-view-root #main-container .col1-container .v-eng_butcher,
#f_eng_bywater:checked  ~ #app-view-root #main-container .col1-container .v-eng_bywater,

#c1_grc_kassel:checked  ~ #app-view-root #main-container .col2-container .v-grc_kassel,
#c1_grc_digi:checked    ~ #app-view-root #main-container .col2-container .v-grc_digi,
#c1_eng_fyfe:checked    ~ #app-view-root #main-container .col2-container .v-eng_fyfe,
#c1_eng_butcher:checked ~ #app-view-root #main-container .col2-container .v-eng_butcher,
#c1_eng_bywater:checked ~ #app-view-root #main-container .col2-container .v-eng_bywater,

#c2_grc_kassel:checked  ~ #app-view-root #main-container .col3-container .v-grc_kassel,
#c2_grc_digi:checked    ~ #app-view-root #main-container .col3-container .v-grc_digi,
#c2_eng_fyfe:checked    ~ #app-view-root #main-container .col3-container .v-eng_fyfe,
#c2_eng_butcher:checked ~ #app-view-root #main-container .col3-container .v-eng_butcher,
#c2_eng_bywater:checked ~ #app-view-root #main-container .col3-container .v-eng_bywater {
    display: block;
}

/* ── Unified Target-Isolator Sibling Engine ── */
.section-row { border-bottom: 1px solid #f0f0f0; padding: 6px 0; display: block; scroll-margin-top: 25px; }
.section-row .sec-num { font-weight: bold; color: #990000; margin-right: 8px; display: inline-block; width: 30px; }

body:has(.target-marker:target) .section-row { display: none; }

#global_sec_1:target ~ #app-view-root .s-idx-1, #global_sec_2:target ~ #app-view-root .s-idx-2,
#global_sec_3:target ~ #app-view-root .s-idx-3, #global_sec_4:target ~ #app-view-root .s-idx-4,
#global_sec_5:target ~ #app-view-root .s-idx-5, #global_sec_6:target ~ #app-view-root .s-idx-6,
#global_sec_7:target ~ #app-view-root .s-idx-7, #global_sec_8:target ~ #app-view-root .s-idx-8,
#global_sec_9:target ~ #app-view-root .s-idx-9, #global_sec_10:target ~ #app-view-root .s-idx-10,
#global_sec_11:target ~ #app-view-root .s-idx-11, #global_sec_12:target ~ #app-view-root .s-idx-12,
#global_sec_13:target ~ #app-view-root .s-idx-13, #global_sec_14:target ~ #app-view-root .s-idx-14,
#global_sec_15:target ~ #app-view-root .s-idx-15, #global_sec_16:target ~ #app-view-root .s-idx-16,
#global_sec_17:target ~ #app-view-root .s-idx-17, #global_sec_18:target ~ #app-view-root .s-idx-18,
#global_sec_19:target ~ #app-view-root .s-idx-19, #global_sec_20:target ~ #app-view-root .s-idx-20 {
    display: block !important;
}

body:has(.target-marker:target) #app-view-root .tier-indicator::after {
    content: " (Isolated Section Parallel Mode - click 'View Full Chapter' down below to reset)";
    color: #990000;
}

.target-marker { display: none; position: absolute; }

/* Text Typography */
.greek-text { font-size: 15px; line-height: 1.7; font-family: "Gentium Plus", "Athena", serif; color: #000; }
.english-text { font-size: 13px; line-height: 1.6; color: #222; }
.note { font-size: 11px; color: #666; font-style: italic; background: #f4f4f0; padding: 0 2px; }
.tier-indicator { background: #fffccb; border: 1px solid #e6db55; padding: 4px 8px; font-size: 11px; margin-bottom: 5px; font-weight: bold; width: 100%; flex-shrink: 0; }
.reset-btn { display: inline-block; margin-top: 15px; background: #660000; color: #fff !important; padding: 4px 10px; font-size: 11px; font-weight: bold; border-radius: 2px; }
"""

# ── Namespace-Agnostic TEI XML Engine ────────────────────────
def extract_text_recursive(elem):
    parts = []
    if elem.text: parts.append(elem.text)
    for child in elem:
        tag = child.tag.replace('{http://www.tei-c.org/ns/1.0}', '')
        if tag == 'note':
            note_text = extract_text_recursive(child).strip()
            if note_text: parts.append(f'<span class="note">[{note_text}]</span>')
        elif tag in ('milestone', 'reg'): pass
        else: parts.append(extract_text_recursive(child))
        if child.tail: parts.append(child.tail)
    return ''.join(parts)

def parse_tei_hierarchy(path):
    if not os.path.exists(path): return None
    tree = ET.parse(path)
    root = tree.getroot()
    body = root.find('.//tei:body', NS) or root.find('.//body')
    if body is None: return None
    top_div = body.find('tei:div', NS) or body.find('div') or body
    data = OrderedDict()
    textparts = top_div.findall('tei:div', NS) or top_div.findall('div')
    if not textparts:
        textparts = top_div.findall('.//tei:div[@subtype="chapter"]', NS) or top_div.findall('.//div[@subtype="chapter"]')

    if textparts and (textparts[0].get('subtype') == 'chapter' or textparts[0].get('type') == 'chapter'):
        data['1'] = OrderedDict()
        for ch_div in textparts:
            ch_n = ch_div.get('n')
            if not ch_n: continue
            data['1'][ch_n] = OrderedDict()
            sections = ch_div.findall('tei:div[@subtype="section"]', NS) or ch_div.findall('div[@subtype="section"]')
            if not sections:
                sections = ch_div.findall('tei:div', NS) or ch_div.findall('div') or ch_div.findall('tei:p', NS) or ch_div.findall('p')
            for idx, sec_div in enumerate(sections):
                sec_n = sec_div.get('n') or str(idx + 1)
                paragraphs = sec_div.findall('tei:p', NS) or sec_div.findall('p')
                data['1'][ch_n][sec_n] = ' '.join(extract_text_recursive(p).strip() for p in paragraphs) if paragraphs else extract_text_recursive(sec_div).strip()
    return data

print("Parsing absolute system path text data...")
MASTER_CORPUS = OrderedDict()
for v_id, cfg in TEXT_REGISTRY.items():
    parsed = parse_tei_hierarchy(cfg["path"])
    if parsed: 
        MASTER_CORPUS[v_id] = parsed
        print(f" -> Loaded successfully: {cfg['label']}")

baseline_structure = MASTER_CORPUS["grc_kassel"]["1"]
chapter_keys = list(baseline_structure.keys())

def get_chapter_filename(ch_id):
    return f"chapter_{ch_id}.html"

# ── Template Core Component Assembly ──────────────────────────
def render_top_navigation_matrix(current_ch):
    html = ['<div id="browse-bar"><div class="browse-row"><label>Chapters:</label><div class="browse-items">']
    for ck in chapter_keys:
        cls = ' class="current"' if ck == current_ch else ''
        html.append(f'<a href="{get_chapter_filename(ck)}"{cls}>{ck}</a>')
    html.append('</div></div></div>')
    return '\n'.join(html)

def render_sidebar_toc(current_ch):
    html = ['<div id=\"sidebar-toc\"><h3>Poetics Chapters</h3><ul>']
    for ck in chapter_keys:
        cls = ' class="current"' if ck == current_ch else ''
        html.append(f'<li><a href="{get_chapter_filename(ck)}"{cls}>Chapter {ck}</a>')
        if ck == current_ch:
            html.append('<div class="toc-sections">')
            for sk in baseline_structure[ck].keys():
                html.append(f'<a href="#global_sec_{sk}">{sk}</a>')
            html.append('</div>')
        html.append('</li>')
    html.append('</ul></div>')
    return '\n'.join(html)

def render_column_viewport(ch_id, col_class_wrapper, prefix_token, radio_prefix):
    """Outputs the column container complete with its localized native popup selection option set."""
    html = [f'<div class="{col_class_wrapper} reading-column">']
    
    # Render the local selection box panel header container
    html.append(f'<div class="panel-header">')
    html.append(f'  <span>{prefix_token}</span>')
    
    # Embed the native zero-JS choice selector dropdown menu
    html.append(f'  <select class="edition-select-dropdown" onchange="document.getElementById(\'{radio_prefix}_\' + this.value).click();">')
    for v_id, cfg in TEXT_REGISTRY.items():
        # Cleaned up unambiguous logical defaults assignment 
        if radio_prefix == "f" and v_id == "grc_kassel": selected_flag = " selected"
        elif radio_prefix == "c1" and v_id == "eng_fyfe": selected_flag = " selected"
        elif radio_prefix == "c2" and v_id == "eng_bywater": selected_flag = " selected"
        else: selected_flag = ""
        
        html.append(f'    <option value="{v_id}"{selected_flag}>{cfg["label"]}</option>')
    html.append(f'  </select>')
    html.append(f'</div>')
    
    # Populate version blocks
    for v_id in TEXT_REGISTRY:
        html.append(f'<div class="version-block v-{v_id}">')
        ch_data = MASTER_CORPUS.get(v_id, {}).get("1", {}).get(ch_id, {})
        for sk, text in ch_data.items():
            html.append(
                f'<div class="section-row s-idx-{sk}">'
                f'  <span class="sec-num"><a href="#global_sec_{sk}">[{sk}]</a></span>'
                f'  <span class="text-p">{text if text.strip() else "<i>[Text part missing in this version]</i>"}</span>'
                f'</div>'
            )
        html.append('</div>')
    html.append('<a href="#" class="reset-btn">View Full Chapter</a>')
    html.append('</div>')
    return '\n'.join(html)

print(f"Compiling streamlined standalone workspace modules...")
start_time = time.time()

for ch_id in chapter_keys:
    form_inputs = """
    <input type="checkbox" id="toc-toggle">
    <input type="radio" name="focus_col" id="f_grc_kassel" checked>
    <input type="radio" name="focus_col" id="f_grc_digi">
    <input type="radio" name="focus_col" id="f_eng_fyfe">
    <input type="radio" name="focus_col" id="f_eng_butcher">
    <input type="radio" name="focus_col" id="f_eng_bywater">

    
    <input type="radio" name="cross_col1" id="c1_grc_kassel">
    <input type="radio" name="cross_col1" id="c1_grc_digi">
    <input type="radio" name="cross_col1" id="c1_eng_fyfe" checked>
    <input type="radio" name="cross_col1" id="c1_eng_butcher">
    <input type="radio" name="cross_col1" id="c1_eng_bywater">
    
    <input type="radio" name="cross_col2" id="c2_grc_kassel">
    <input type="radio" name="cross_col2" id="c2_grc_digi">
    <input type="radio" name="cross_col2" id="c2_eng_fyfe">
    <input type="radio" name="cross_col2" id="c2_eng_butcher" >
    <input type="radio" name="cross_col2" id="c2_eng_bywater" checked>
    """
    
    marker_tags = []
    for i in range(1, 21):
        marker_tags.append(f'<div class="target-marker" id="global_sec_{i}"></div>')
    
    master_html = f"""<!DOCTYPE html>
<html>
<head><meta charset="UTF-8"><title>Aristotle Poetics - Chapter {ch_id}</title><style>{PERSEUS_CSS}</style></head>
<body>
  {form_inputs}
  {''.join(marker_tags)}

  <div id="app-view-root">
    <div id="header-container">
      <div id="perseus-banner">
        <h1><a href="chapter_1.html">Perseus Digital Library</a></h1>
        <span class="doc-title">Aristotle, Poetics Parallel Workspace</span>
      </div>
      <div id="nav-bar">
        <label for="toc-toggle" class="toc-toggle-btn"><span class="show-lbl">Show TOC</span><span class="hide-lbl">Hide TOC</span></label>
        <a href="chapter_1.html">Home Workspace</a>
      </div>
      {render_top_navigation_matrix(ch_id)}
      <div class="tier-indicator">
         Active Work Frame Context: Aristotle, Poetics, Chapter {ch_id}
      </div>
    </div>

    <div id="outer-wrapper">
      {render_sidebar_toc(ch_id)}
      <div id="main-container">
        {render_column_viewport(ch_id, "col1-container", "Focal Column", "f")}
        {render_column_viewport(ch_id, "col2-container cross-panel", "Comparison Column 1", "c1")}
        {render_column_viewport(ch_id, "col3-container cross-panel", "Comparison Column 2", "c2")}
      </div>
    </div>
  </div>
</body>
</html>"""
    
    (OUT_DIR / get_chapter_filename(ch_id)).write_text(master_html, encoding='utf-8')

print(f"\n[SUCCESS] Successfully compiled the unified multi-version dropdown workspace layout.")

Parsing absolute system path text data...
 -> Loaded successfully: Greek (Kassel, 1965)
 -> Loaded successfully: Greek (Digital Corpus Variant)
 -> Loaded successfully: English (W.H. Fyfe, 1927)
 -> Loaded successfully: English (S.H. Butcher, 1911)
 -> Loaded successfully: English (Ingram Bywater, 1909)
Compiling streamlined standalone workspace modules...

[SUCCESS] Successfully compiled the unified multi-version dropdown workspace layout.
